# Group 1 — Readiness (heart & stress)

Load your Delsys recording, find the heartbeats, and look at **heart-rate variability (HRV)**. You decide the calm / stress / breathing windows.

Your sensors: **Ch 1 = right trapezius EMG**, **Ch 12 = EKG (chest, + IMU)**.

In [ ]:
!pip install -q delsys pysampled huggingface_hub

### 1 · Get your data

In [ ]:
from huggingface_hub import hf_hub_download
REPO = "praneethnamburimit/immersionlab-pe-mis-groups"
csv = hf_hub_download(REPO, "group_1/Trial_1.csv", repo_type="dataset")   # you also have Trial_2.csv
print(csv)

### 2 · Load it
`delsys.Log` reads the Trigno export. Print it to see the sensors inside.

In [ ]:
import delsys
lf = delsys.Log(csv)
lf

### 3 · Look at the EKG

In [ ]:
import matplotlib.pyplot as plt
ekg = lf.find(sensor_number=12, as_="sensor")[0].ekg   # Ch 12 = EKG (single channel)
plt.figure(figsize=(12,3)); plt.plot(ekg.t, ekg()); plt.xlim(20,30)
plt.xlabel("s"); plt.title("EKG (20–30 s)"); plt.show()

### 4 · Find the heartbeats (R-peaks)
`find_rpeaks()` returns the sample index of each beat.

In [ ]:
peaks = ekg.find_rpeaks()
tp = ekg.t[peaks]
plt.figure(figsize=(12,3)); plt.plot(ekg.t, ekg()); plt.plot(tp, ekg()[peaks], "r.", ms=9)
plt.xlim(20,30); plt.title(f"{len(peaks)} beats detected"); plt.show()
print(len(peaks), "beats")

> **Manual correction:** Praneeth will open this in the **EKGBrowser** to fix any missed / extra beats before you trust the HRV numbers.

### 5 · Inter-beat intervals + RMSSD

In [ ]:
import numpy as np
rr = np.diff(tp)*1000.0            # inter-beat intervals, ms
plt.figure(figsize=(12,3)); plt.plot(tp[1:], rr, "-o", ms=3)
plt.ylabel("IBI (ms)"); plt.xlabel("s"); plt.title("inter-beat intervals"); plt.show()
rmssd = np.sqrt(np.mean(np.diff(rr)**2))
print(f"RMSSD (whole recording) = {rmssd:.1f} ms")

### 6 · Your turn
- Split the recording into your **calm → stress → breathing** windows (you know when each happened) and compute RMSSD in each. Higher RMSSD ≈ calmer.
- Also try the **trapezius EMG**: `emg = lf.find(sensor_number=1, as_="sensor")[0].emg` then `emg.bandpass(20,450).envelope()` — does muscle tension rise under stress?
- `ekg.get_features_hp(win_size=30.0, win_inc=1.0)` gives windowed HRV (rmssd, sdnn, bpm) over time.